<a href="https://colab.research.google.com/github/Vonna9/CallShield/blob/AI-Detection/AITraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook aims to take a specific data set from HuggingFace.co, which has both AI and human voice samples and train itself without the need to annoyingly download 'torchodec'.

**Why this?**

Initially, training the model indicated that we would install 'torchodec', which is annoying and takes up about 2 GB of space. In addition, I was using a loaner MacBook, so installing Homebrew was not possible. (Context: Installing Homebrew would have guaranteed that I would have downloaded PyTorch, which includes torchodec.)

Other solutions were brought up, such as downloading Python 3.13, but the issue persisted.

# Installing Dependencies

In [ ]:
!pip install datasets soundfile torch torchaudio

# Download Dataset

In [ ]:
from datasets import load_dataset
import soundfile as sf
import numpy as np
import os

os.makedirs("data/human", exist_ok=True)
os.makedirs("data/ai", exist_ok=True)

print("Loading dataset...")
ds = load_dataset("garystafford/deepfake-audio-detection", split="train")

human_count = 0
ai_count = 0

for i in range(len(ds)):
    try:
        sample = ds[i]
        audio = sample["audio"]
        y = np.array(audio["array"], dtype=np.float32)
        sr = audio["sampling_rate"]
        label = sample["label"]

        if label == 0 and human_count < 200:
            path = f"data/human/human_{human_count:04d}.wav"
            sf.write(path, y, sr)
            human_count += 1
            if human_count % 20 == 0:
                print(f"✓ Human: {human_count}")

        elif label == 1 and ai_count < 200:
            path = f"data/ai/ai_{ai_count:04d}.wav"
            sf.write(path, y, sr)
            ai_count += 1
            if ai_count % 20 == 0:
                print(f"✓ AI: {ai_count}")

        if human_count >= 200 and ai_count >= 200:
            break

    except Exception as e:
        continue

print(f"\nDone! {human_count} human + {ai_count} AI samples saved.")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/114M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1866 [00:00<?, ? examples/s]

✓ Human: 20
✓ Human: 40
✓ Human: 60
✓ Human: 80
✓ Human: 100
✓ Human: 120
✓ Human: 140
✓ Human: 160
✓ Human: 180
✓ Human: 200
✓ AI: 20
✓ AI: 40
✓ AI: 60
✓ AI: 80
✓ AI: 100
✓ AI: 120
✓ AI: 140
✓ AI: 160
✓ AI: 180
✓ AI: 200

Done! 200 human + 200 AI samples saved.


#Build the Dataset

In [ ]:
import numpy as np
import soundfile as sf
import scipy.fft as fft
import os
import csv

def extract_features(audio_path):
    y, sr = sf.read(audio_path, always_2d=False)
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)
    y = y.astype(np.float32)

    spectral_data = np.abs(fft.fft(y))
    freqs = fft.fftfreq(len(spectral_data), 1/sr)
    hi_freq = spectral_data[(freqs > 10000) & (freqs < 15000)]
    fft_score = float(np.mean(hi_freq) / np.mean(spectral_data)) if len(hi_freq) > 0 else 0.0

    frame_size = 512
    frames = [y[i:i+frame_size] for i in range(0, len(y)-frame_size, frame_size//2)]
    energies = np.array([np.sum(f**2) for f in frames])
    delta_var = float(np.var(np.diff(energies)))

    zero_crossing = float(np.mean(np.abs(np.diff(np.sign(y)))) / 2)
    spectral_centroid = float(np.sum(freqs[:len(freqs)//2] * spectral_data[:len(spectral_data)//2]) / np.sum(spectral_data[:len(spectral_data)//2]))
    rms_energy = float(np.sqrt(np.mean(y**2)))

    return [fft_score, delta_var, zero_crossing, spectral_centroid, rms_energy]

data = []

for label, folder in [("human", "data/human"), ("ai", "data/ai")]:
    for filename in os.listdir(folder):
        if filename.endswith(".wav"):
            path = os.path.join(folder, filename)
            try:
                features = extract_features(path)
                data.append(features + [label])
            except Exception as e:
                print(f"✗ Skipped {filename}: {e}")

with open("training_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["fft_score", "delta_var", "zero_crossing", "spectral_centroid", "rms_energy", "label"])
    writer.writerows(data)

print(f"Done! Saved {len(data)} samples to training_data.csv")

Done! Saved 400 samples to training_data.csv


# Train the Model

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

df = pd.read_csv("training_data.csv")
print(f"Loaded {len(df)} samples")
print(f"  Human: {len(df[df['label'] == 'human'])}")
print(f"  AI:    {len(df[df['label'] == 'ai'])}")

X = df[["fft_score", "delta_var", "zero_crossing", "spectral_centroid", "rms_energy"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("\n===== ACCURACY REPORT =====")
print(classification_report(y_test, model.predict(X_test)))

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("model.pkl saved!")

Loaded 400 samples
  Human: 200
  AI:    200

===== ACCURACY REPORT =====
              precision    recall  f1-score   support

          ai       1.00      1.00      1.00        36
       human       1.00      1.00      1.00        44

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

model.pkl saved!


#  Download model.pkl to your computer

In [ ]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>